In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
import numpy as np
import pathlib
import seaborn as sns

In [2]:
manifest_path = "/home/furkan/projects/mebar-radiology/data/manifest.csv"
manifest_df = pd.read_csv(manifest_path)

### Load dataset

In [3]:
print(manifest_df.columns)
manifest_df.head()

Index(['sample_id', 'batch_id', 'response_group', 'patient_name', 'patient_id',
       'modality', 'patient_path', 'modality_path', 'nesting_depth',
       'image_path', 'mask_path', 'image_filename', 'mask_filename',
       'nrrd_file_count', 'other_file_count', 'image_candidate_count',
       'mask_candidate_count', 'image_dimension', 'image_shape_x',
       'image_shape_y', 'image_shape_z', 'image_dtype', 'image_encoding',
       'image_spacing_x_mm', 'image_spacing_y_mm', 'image_spacing_z_mm',
       'image_origin', 'mask_dimension', 'mask_shape_x', 'mask_shape_y',
       'mask_shape_z', 'mask_dtype', 'mask_encoding', 'mask_spacing_x_mm',
       'mask_spacing_y_mm', 'mask_spacing_z_mm', 'mask_origin',
       'shape_matches', 'spacing_matches', 'origin_matches', 'is_valid_sample',
       'validation_errors', 'manifest_created_at'],
      dtype='str')


,sample_id,batch_id,response_group,patient_name,patient_id,modality,patient_path,modality_path,nesting_depth,image_path,...,mask_spacing_x_mm,mask_spacing_y_mm,mask_spacing_z_mm,mask_origin,shape_matches,spacing_matches,origin_matches,is_valid_sample,validation_errors,manifest_created_at
0,0d6f73a01539ae0d,batch-1,intermediate,"ABDULLAEVA, GULSARA",a28eea2bdc22b7bc,ADC,"data/raw/batch-1/İNTERMEDİATE/ABDULLAEVA, GU...","data/raw/batch-1/İNTERMEDİATE/ABDULLAEVA, GU...",0,"data/raw/batch-1/İNTERMEDİATE/ABDULLAEVA, GU...",...,1.785714,1.785714,6.600000,"(-183.96600185888997,-168.05404737164,-244.827...",True,True,True,True,NaN,2026-08-11T12:27:41.449930+00:00
1,85388bd87a565011,batch-1,intermediate,"ABDULLAEVA, GULSARA",a28eea2bdc22b7bc,T2,"data/raw/batch-1/İNTERMEDİATE/ABDULLAEVA, GU...","data/raw/batch-1/İNTERMEDİATE/ABDULLAEVA, GU...",0,"data/raw/batch-1/İNTERMEDİATE/ABDULLAEVA, GU...",...,0.397727,0.397727,3.500000,"(-54.253527797160011,13.519029497832603,-212.4...",True,True,True,True,NaN,2026-08-11T12:27:41.449930+00:00
2,c5574c732708c9c0,batch-1,intermediate,"AKKAYA, HATICE",c76cde132a53d60f,ADC,"data/raw/batch-1/İNTERMEDİATE/AKKAYA, HATICE","data/raw/batch-1/İNTERMEDİATE/AKKAYA, HATICE...",0,"data/raw/batch-1/İNTERMEDİATE/AKKAYA, HATICE...",...,0.781250,0.781250,3.600031,"(-26.884099999999997,10.567199999999977,-337.0...",True,True,True,True,NaN,2026-08-11T12:27:41.449930+00:00
3,d4e511f387bc2059,batch-1,intermediate,"AKKAYA, HATICE",c76cde132a53d60f,T2,"data/raw/batch-1/İNTERMEDİATE/AKKAYA, HATICE","data/raw/batch-1/İNTERMEDİATE/AKKAYA, HATICE/T2",0,"data/raw/batch-1/İNTERMEDİATE/AKKAYA, HATICE...",...,0.595238,0.595238,3.600029,"(-76.883799999999994,10.620900000000018,-337.2...",True,True,True,True,NaN,2026-08-11T12:27:41.449930+00:00
4,85e3e9f712fe9029,batch-1,intermediate,"ARSLAN, AYSEL",fd5ae5ecb097ff8e,ADC,"data/raw/batch-1/İNTERMEDİATE/ARSLAN, AYSEL","data/raw/batch-1/İNTERMEDİATE/ARSLAN, AYSEL/ADC",0,"data/raw/batch-1/İNTERMEDİATE/ARSLAN, AYSEL/...",...,1.785714,1.785714,6.600000,"(-207.14045924210006,-168.23679757503001,-288....",True,True,True,True,NaN,2026-08-11T12:27:41.449930+00:00


### Null and Duplicate check

In [4]:
manifest_df.isnull().sum()

# No null!

sample_id                  0
batch_id                   0
response_group             0
patient_name               0
patient_id                 0
modality                   0
patient_path               0
modality_path              0
nesting_depth              0
image_path                 0
mask_path                  0
image_filename             0
mask_filename              0
nrrd_file_count            0
other_file_count           0
image_candidate_count      0
mask_candidate_count       0
image_dimension            0
image_shape_x              0
image_shape_y              0
image_shape_z              0
image_dtype                0
image_encoding             0
image_spacing_x_mm         0
image_spacing_y_mm         0
image_spacing_z_mm         0
image_origin               0
mask_dimension             0
mask_shape_x               0
mask_shape_y               0
mask_shape_z               0
mask_dtype                 0
mask_encoding              0
mask_spacing_x_mm          0
mask_spacing_y

In [5]:
manifest_df[manifest_df.duplicated()==True]

# No duplicate !

,sample_id,batch_id,response_group,patient_name,patient_id,modality,patient_path,modality_path,nesting_depth,image_path,...,mask_spacing_x_mm,mask_spacing_y_mm,mask_spacing_z_mm,mask_origin,shape_matches,spacing_matches,origin_matches,is_valid_sample,validation_errors,manifest_created_at


### Label - Modality matrix

In [6]:
labels = manifest_df["response_group"].unique().tolist()
modalities = manifest_df["modality"].unique().tolist()

matrix = pd.crosstab(
    manifest_df["modality"],
    manifest_df["response_group"],
    margins=True,
    margins_name="total"
)

matrix

response_group,intermediate,nonresponder,responder,total
modality,,,,
ADC,52,44,92,188
T2,52,44,92,188
total,104,88,184,376
